# 介紹

使用 `pyautogui` 偵測 Chrome 小恐龍前方的障礙物並自動跳躍。

小恐龍是在 Chrome 中使用 `chrome://dino` 啟動。執行前請先開啟遊戲，並將瀏覽器視窗放在螢幕上。

偵測座標以螢幕左上角為原點；支援 Chrome 深色模式。若遊戲視窗移動或縮放，請調整 `DETECTION_REGION`。按 Ctrl+C 可停止程式，或在 Jupyter 中按停止按鈕。

In [61]:
import time
import warnings
import pyautogui as pyg

warnings.filterwarnings('ignore', category=DeprecationWarning)

# 【速度自適應調整】
# 當遊戲分數變高、速度加快時，原本的偵測區會來不及反應。
# 我們將偵測框的寬度從 90 拉寬到 140，讓恐龍更早（距離更遠）看到障礙物提早起跳！
DETECTION_REGION = (110, 480, 140, 85)
LOOP_DELAY = 0.005  # 縮短迴圈延遲，提升每秒偵測幀數（FPS），應對高速奔跑

def get_average_brightness(image):
    total = 0
    pixels = list(image.convert('RGB').getdata())
    for red, green, blue in pixels:
        total += (red + green + blue) / 3
    return total / len(pixels)

pyg.PAUSE = 0
print('請在 2 秒內點擊切換到遊戲視窗...')
pyg.sleep(2)
pyg.press('space')  # 開始遊戲

# 記錄背景基準亮度
base_brightness = get_average_brightness(pyg.screenshot(region=DETECTION_REGION))
print(f'背景基準亮度: {base_brightness:.2f}')

was_obstacle_near = False
print('高速自適應恐龍機器人運行中...')
try:
    while True:
        screen = pyg.screenshot(region=DETECTION_REGION)
        current_brightness = get_average_brightness(screen)
        
        brightness_diff = current_brightness - base_brightness
        
        # 門檻保持 1.5
        obstacle_near = brightness_diff > 1.5

        print(f'亮度差: {brightness_diff:.2f} | 判定跳躍: {obstacle_near}', end='\r')

        if obstacle_near and not was_obstacle_near:
            pyg.press('space')
        was_obstacle_near = obstacle_near
        time.sleep(LOOP_DELAY)
except KeyboardInterrupt:
    print('\n使用者手動停止。')
finally:
    pyg.keyUp('space')
    print('恐龍機器人已停止。')

請在 2 秒內點擊切換到遊戲視窗...
背景基準亮度: 37.18
高速自適應恐龍機器人運行中...
亮度差: 142.21 | 判定跳躍: True
使用者手動停止。
恐龍機器人已停止。


In [ ]:
# 檢查螢幕尺寸與指定座標的像素顏色，可用來校準偵測區域。
import pyautogui as pyg

sample_point = (110, 540)  # 改成想檢查的螢幕座標
screen = pyg.screenshot()
print(f'螢幕大小：{screen.size}')
print(f'{sample_point} 的 RGB 顏色：{screen.getpixel(sample_point)}')

# 【即時滑鼠座標與顏色追蹤器】
import time

print('開始追蹤滑鼠位置（按 Jupyter 的停止按鈕或 Interrupt 即可結束）...')
try:
    while True:
        x, y = pyg.position()
        try:
            pixel_color = pyg.screenshot().getpixel((x, y))
        except Exception:
            pixel_color = (0, 0, 0)
        print(f'目前滑鼠座標: x={x}, y={y} | RGB 顏色: {pixel_color}', end='\r')
        time.sleep(0.1)
except KeyboardInterrupt:
    print('\n已停止座標追蹤。')
    print(f'最終滑鼠座標: x={x}, y={y}')